# AutoML Test — benchmark exploratorio

Laboratorio para comparar familias de modelos sobre la misma tabla integrada. **No sustituye a
`Models.ipynb`**: no guarda artefactos, no promociona nada y ningún resultado de aquí cambia el
modelo de producción.

Cómo leerlo:

- Todos los modelos predicen lo mismo que el principal: la **fracción del plazo** que sobrevive el
  producto. El MAE se reporta en meses, después de multiplicar por el plazo.
- Se elige configuración mirando **solo 2023**, y las tres mejores se comprueban en 2024.
- Nunca hay división aleatoria: todo respeta el orden temporal.
- Un MAE menor no basta para adoptar un modelo. Habría que mirar estabilidad entre ventanas,
  explicabilidad y sentido de negocio — y con este tamaño de muestra, diferencias de un par de
  décimas de mes son ruido.

## 1. Setup y contrato

Se lee el mismo contrato de features que usan el modelo principal y la API, para que la comparación
sea justa y no se cuele ninguna variable que en producción no existe.

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import (
    ExtraTreesRegressor, GradientBoostingRegressor,
    HistGradientBoostingRegressor, RandomForestRegressor,
)
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.svm import SVR

RANDOM_SEED = 42
TARGET = "avg_duration_months"
MATURITY = "nominal_maturity_months"

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data" / "processed").exists() else cwd.parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

data = pd.read_csv(PROCESSED_DIR / "train_features.csv", parse_dates=["requested_date"])
feature_columns = pd.read_csv(PROCESSED_DIR / "model_feature_columns.csv")["feature_name"].tolist()
data["duration_ratio"] = data[TARGET] / data[MATURITY]

assert data["executed"].all()
assert data[TARGET].notna().all()
assert not data[feature_columns].isna().any().any()

print(f"{len(data):,} RFQs ejecutadas y {len(feature_columns)} variables del contrato.")

## 2. Protocolo temporal

2023 para elegir, 2024 para comprobar. El periodo de 2024 ya se usó en `Models.ipynb`, así que aquí
cuenta como comprobación exploratoria, no como un resultado independiente de producción.

In [ ]:
validation_start = pd.Timestamp("2023-01-01")
test_start = pd.Timestamp("2024-01-01")

train = data.loc[data["requested_date"] < validation_start].copy()
validation = data.loc[(data["requested_date"] >= validation_start) & (data["requested_date"] < test_start)].copy()
test = data.loc[data["requested_date"] >= test_start].copy()

assert train.requested_date.max() < validation.requested_date.min() < test.requested_date.min()

print(pd.DataFrame({
    "split": ["train", "validation", "test exploratorio"],
    "n_rfqs": [len(train), len(validation), len(test)],
    "desde": [train.requested_date.min().date(), validation.requested_date.min().date(), test.requested_date.min().date()],
    "hasta": [train.requested_date.max().date(), validation.requested_date.max().date(), test.requested_date.max().date()],
}).to_string(index=False))

## 3. Escalado

Los modelos lineales y los de distancia necesitan variables en escalas comparables. Los árboles no:
sus particiones no dependen de que una variable esté en créditos, meses o porcentaje.

Para los lineales se escalan solo las continuas y se dejan las dummies como están; para los de
distancia (KNN, SVR) se escala todo, porque la distancia sí mezcla ambas.

In [ ]:
DUMMY_PREFIXES = ("has_underlying_", "product_type_", "basket_type_", "counterparty_", "trader_id_")
continuous_columns = [c for c in feature_columns if not c.startswith(DUMMY_PREFIXES)]
dummy_columns = [c for c in feature_columns if c.startswith(DUMMY_PREFIXES)]

def scale_continuous_only(scaler) -> ColumnTransformer:
    """Escala las continuas y deja intactas las dummies."""
    return ColumnTransformer([
        ("continuous", scaler, continuous_columns),
        ("dummies", "passthrough", dummy_columns),
    ], remainder="drop")

print(f"Continuas: {len(continuous_columns)} | Dummies: {len(dummy_columns)}")

## 4. Parrilla de configuraciones

No es una búsqueda infinita: es una parrilla pequeña, explícita y reproducible, con una entrada por
familia de modelo. La primera es el baseline estructural del proyecto — predecir el ratio medio
equivale a decir *"todo producto vive la misma fracción de su plazo"*.

In [ ]:
configs = [
    ("Baseline estructural (ratio medio)", lambda: DummyRegressor(strategy="mean")),
    ("Ridge standard", lambda: Pipeline([
        ("preprocess", scale_continuous_only(StandardScaler())), ("model", Ridge(alpha=10.0))])),
    ("Ridge robust", lambda: Pipeline([
        ("preprocess", scale_continuous_only(RobustScaler())), ("model", Ridge(alpha=10.0))])),
    ("ElasticNet", lambda: Pipeline([
        ("preprocess", scale_continuous_only(StandardScaler())),
        ("model", ElasticNet(alpha=0.03, l1_ratio=0.15, max_iter=10_000, random_state=RANDOM_SEED))])),
    ("KNN", lambda: Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=30, weights="distance", p=2, n_jobs=-1))])),
    ("SVR RBF", lambda: Pipeline([
        ("scale", MinMaxScaler()), ("model", SVR(C=10.0, epsilon=0.2, gamma="scale"))])),
    ("Random forest", lambda: RandomForestRegressor(
        n_estimators=350, min_samples_leaf=3, max_features=0.7, random_state=RANDOM_SEED, n_jobs=-1)),
    ("Extra trees", lambda: ExtraTreesRegressor(
        n_estimators=350, min_samples_leaf=2, max_features=0.8, random_state=RANDOM_SEED, n_jobs=-1)),
    ("Gradient boosting", lambda: GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.04, max_depth=3, min_samples_leaf=8,
        loss="huber", random_state=RANDOM_SEED)),
    ("Histogram gradient boosting", lambda: HistGradientBoostingRegressor(
        max_iter=350, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=2.0,
        random_state=RANDOM_SEED)),
    ("CatBoost", lambda: CatBoostRegressor(
        loss_function="MAE", eval_metric="MAE", iterations=800, learning_rate=0.05,
        depth=8, l2_leaf_reg=5.0, random_seed=RANDOM_SEED, verbose=False, allow_writing_files=False)),
]

print(f"{len(configs)} configuraciones a comparar.")

## 5. Selección con 2023

Cada modelo se entrena sobre el ratio y su predicción se devuelve a meses multiplicando por el
plazo, para que el MAE sea comparable con el resto del proyecto.

In [ ]:
def evaluate_configs(configs, train_frame, eval_frame, label: str) -> pd.DataFrame:
    """Entrena sobre el ratio y devuelve el error en meses."""
    cap = float(train_frame["duration_ratio"].max())      # techo calculado solo con train
    y_true = eval_frame[TARGET]
    maturity = eval_frame[MATURITY].to_numpy()
    rows = []
    for name, factory in configs:
        started = time.perf_counter()
        try:
            model = factory()
            model.fit(train_frame[feature_columns], train_frame["duration_ratio"])
            ratio = np.clip(model.predict(eval_frame[feature_columns]), 0.0, cap)
            prediction = ratio * maturity
            rows.append({
                "modelo": name, "estado": "ok",
                f"{label}_MAE_meses": mean_absolute_error(y_true, prediction),
                f"{label}_RMSE_meses": root_mean_squared_error(y_true, prediction),
                "segundos": time.perf_counter() - started,
            })
        except Exception as error:
            rows.append({"modelo": name, "estado": f"error: {type(error).__name__}",
                         "segundos": time.perf_counter() - started})
    return pd.DataFrame(rows)

validation_results = (
    evaluate_configs(configs, train, validation, "validation")
    .sort_values("validation_MAE_meses", na_position="last")
    .reset_index(drop=True)
)
print(validation_results.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## 6. Comprobación de las tres mejores en 2024

Se reentrenan con todo el histórico hasta el 31 de diciembre de 2023 y se miden en 2024.

In [ ]:
mejores = validation_results.loc[validation_results["estado"].eq("ok"), "modelo"].head(3).tolist()
top_configs = [(name, factory) for name, factory in configs if name in mejores]
train_plus_validation = data.loc[data["requested_date"] < test_start].copy()

test_results = (
    evaluate_configs(top_configs, train_plus_validation, test, "test")
    .sort_values("test_MAE_meses", na_position="last")
    .reset_index(drop=True)
)
print(test_results.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print()
print("Ranking exploratorio. Ningún resultado de este notebook modifica los artefactos de models/.")

## Qué hacer con estos resultados

Sirven para saciar curiosidad y encontrar candidatos, no para promocionar modelos. Si alguno
apareciera claramente mejor, el siguiente paso **no** sería adoptarlo: habría que repetir la
comparación con el backtest de origen deslizante de `Models.ipynb`, revisar qué variables usa y
comprobar que sigue teniendo sentido de negocio.

Y antes de leer nada como una mejora: con este tamaño de holdout el error estándar del MAE ronda el
medio mes. Los modelos que se separan por menos que eso están empatados.